# NYC City Jobs, asked in plain English

- **Elastic** stores and searches 1,384 NYC city government job postings, and embeds each one through Mistral (`mistral-embed`) so it can search by meaning.
- **Mistral** reads your question, picks the Elastic query and its filters, then writes the answer from what came back.
- Flow: **Human -> Python -> Mistral -> Elastic -> Mistral -> screen**

## Setup
Connect to Elastic and load the two halves: ingest (index, client) and agent (tools, `ask`).

In [2]:
from dotenv import load_dotenv
load_dotenv()

from nyc_jobs_ingest import INDEX, EMBED_ID, CHAT_ID, es_client
from nyc_jobs_agent import search_jobs, count_jobs, ask

es = es_client()
print(f"{INDEX}: {es.count(index=INDEX)['count']} postings")

nyc_jobs: 1384 postings


## Run once to load data
Already loaded; rerunning empties the index for 5 minutes.

In [3]:
# from nyc_jobs_ingest import main; main()

## Elastic alone: search by meaning
A semantic query on `search_text`. No keyword has to match.

In [4]:
hits = search_jobs(es, "write python backends for city agencies", k=5)

def money(j):
    lo, hi = j["salary_range_from"] or 0, j["salary_range_to"] or 0
    return f"${lo:,.0f} - ${hi:,.0f} {j['salary_frequency'] or ''}"

print(f"{'TITLE':50} {'AGENCY':35} SALARY")
for j in hits:
    print(f"{(j['business_title'] or '')[:49]:50} {(j['agency'] or '')[:34]:35} {money(j)}")

TITLE                                              AGENCY                              SALARY
Web Developer                                      DEPT OF RECORDS & INFO SERVICE      $105,000 - $105,000 Annual
Application Developer                              OFFICE OF THE MAYOR                 $100,000 - $135,000 Annual
Appian Software Developer                          DEPARTMENT OF FINANCE               $125,000 - $150,000 Annual
Application Developer                              DEPT OF CITYWIDE ADMIN SVCS         $99,209 - $114,090 Annual
Budget Research Analyst                            OFFICE OF THE COMPTROLLER           $70,000 - $80,000 Annual


## Elastic alone: counts
Full-time postings whose top salary is at least $100k, with aggregations.

In [5]:
import json
print(json.dumps(count_jobs(es, min_salary=100000, full_time="F"), indent=2))

{
  "total": 551,
  "by_agency": [
    {
      "agency": "DEPT OF ENVIRONMENT PROTECTION",
      "count": 85
    },
    {
      "agency": "DEPT OF HEALTH/MENTAL HYGIENE",
      "count": 68
    },
    {
      "agency": "DEPARTMENT OF TRANSPORTATION",
      "count": 54
    },
    {
      "agency": "OFFICE OF THE MAYOR",
      "count": 42
    },
    {
      "agency": "DEPT OF DESIGN & CONSTRUCTION",
      "count": 39
    },
    {
      "agency": "HRA/DEPT OF SOCIAL SERVICES",
      "count": 32
    },
    {
      "agency": "HOUSING PRESERVATION & DVLPMNT",
      "count": 22
    },
    {
      "agency": "BRONX DISTRICT ATTORNEY",
      "count": 20
    },
    {
      "agency": "LAW DEPARTMENT",
      "count": 15
    },
    {
      "agency": "DEPARTMENT OF CITY PLANNING",
      "count": 12
    }
  ],
  "salary": {
    "count": 551,
    "min": 100000.0,
    "max": 277671.0,
    "avg": 140527.99,
    "sum": 77430922.0
  },
  "by_career_level": [
    {
      "career_level": "Experienced (non-man

## Mistral alone
One plain chat call, no Elastic.

In [6]:
import os
from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
r = client.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user", "content": "In one sentence, what is a civil service title?"}],
)
print(r.choices[0].message.content)

A **civil service title** is an official designation given to a government employee, reflecting their position, rank, and responsibilities within the public sector.


## Both: the agent
Watch the `[Mistral -> Elastic]` lines: Mistral chooses which Elastic query to run and with which filters.

In [7]:
answer = ask("Which city jobs would suit a software engineer who also teaches dance? Pay over 100k.", es=es, client=client)

Q: Which city jobs would suit a software engineer who also teaches dance? Pay over 100k.
[model] mistral-large-4
[Mistral -> Elastic] search_jobs({"query": "software engineer developer programmer coding", "min_salary": 100000, "full_time": "F", "k": 5})
[Elastic -> Mistral] 5 postings
[Mistral -> Elastic] search_jobs({"query": "dance teacher instructor teaching dance classes arts education", "min_salary": 100000, "full_time": "F", "k": 5})
[Elastic -> Mistral] 5 postings
[Mistral -> Elastic] search_jobs({"k": 5, "min_salary": 100000, "query": "dance instructor teacher performing arts recreation youth program community arts education specialist"})
[Elastic -> Mistral] 5 postings
[answered by] mistral-large-4

Based on the postings, here's what fits your profile — note that **no single posting combines software engineering and dance teaching**, so I've split the results by which skill each role uses:

**Software engineering roles (pay over $100k, full-time):**
- **Senior Software Enginee

Counting question: Mistral should pick `count_jobs` instead of `search_jobs`.

In [8]:
answer = ask("How many full-time postings pay over 100k, and which agencies post the most?", es=es, client=client)

Q: How many full-time postings pay over 100k, and which agencies post the most?
[model] mistral-large-4
[Mistral -> Elastic] count_jobs({"full_time": "F", "min_salary": 100000})
[Elastic -> Mistral] 551
[answered by] mistral-large-4

**551** full-time postings pay over $100k.

Top agencies by count:
1. Dept of Environment Protection — 85
2. Dept of Health/Mental Hygiene — 68
3. Department of Transportation — 54
4. Office of the Mayor — 42
5. Dept of Design & Construction — 39
6. HRA/Dept of Social Services — 32
7. Housing Preservation & Dvlpmnt — 22
8. Bronx District Attorney — 20
9. Law Department — 15
10. Department of City Planning — 12

Salary range across these: $100,000–$277,671, average ~$140,528. Most are experienced (non-manager) roles (400), with 126 manager and 21 executive postings.


A question the data only partly answers: Mistral should say which part is not in the postings.

In [9]:
answer = ask("Any part-time or hourly roles that end by 5pm and are in Queens?", es=es, client=client)

Q: Any part-time or hourly roles that end by 5pm and are in Queens?
[model] mistral-large-4
[Mistral -> Elastic] search_jobs({"query": "part-time hourly job in Queens ending by 5pm weekday afternoon shift", "full_time": "P", "k": 8})
[Elastic -> Mistral] 8 postings
[Mistral -> Elastic] search_jobs({"query": "hourly job Queens afternoon ends 5pm", "k": 8})
[Elastic -> Mistral] 8 postings
[Mistral -> Elastic] search_jobs({"query": "part-time hourly Queens work location afternoon shift schedule 5pm end time", "full_time": "P", "k": 8})
[Elastic -> Mistral] 8 postings
[answered by] mistral-large-4

Here are the part-time/hourly postings in Queens from the search results:

**Part-time, hourly, Queens-based:**
1. **Case Management Nurse** — Police Department — $59.93/Hourly — 1 Lefrak City, Queens (Job ID 768800)
2. **Timekeeping Analyst (Part-Time)** — Department of Correction — $25.45–$29.27/Hourly — 75-20 Astoria Blvd, Queens (Job ID 802634)
3. **College Aide** — Dept of Design & Construc

## Try your own

In [10]:
answer = ask("", es=es, client=client)